In [6]:
class Machine_Parallele:
    def __init__(
    self,
    taskInfo, 
    tasks, 
    machines 
    ):
        self.taskInfo = taskInfo
        self.tasks = tasks 
        self.machines = machines

# Create a new CP-SAT model
        self.model = cp_model.CpModel()


# Create start time variables for each task
        self.start_time_vars = {
            task_name: self.model.new_int_var_from_domain(
                cp_model.Domain.from_intervals([[task_info.relase_date, task_info.due_date-task_info.duration]]),
                f"start_{task_name}",
            )
            for task_name, task_info in self.tasks.items()
        }


        self.machine_vars = {
            name: {machine: self.model.new_bool_var(f"{name}_in_{machine}") for machine in self.machines}
            for name in self.tasks
        }

        # Create interval variables and add no-overlap constraint
        self.interval_vars = {
            task_name: {
                # We need a separate interval for each machine
                machine: self.model.new_optional_fixed_size_interval_var(
                    start=self.start_time_vars[task_name],
                    size=task_info.duration,
                    is_present = self.machine_vars[task_name][machine],
                    name=f"interval_{task_name}_in_{machine}",
                )
                for machine in self.machines
            }
            for task_name, task_info in self.tasks.items()
        }


            # Ensure each task is assigned to exactly one machine
        for name, machine_dict in self.machine_vars.items():
            self.model.add_exactly_one(machine_dict.values())

        for machine in self.machines:
            self.model.add_no_overlap([self.interval_vars[task_name][machine] for task_name in self.tasks])



            # Ensure the precedence constraints
        for task_name, task_info in self.tasks.items():
            if task_info.predecessors != "none":
                self.model.Add(self.start_time_vars[task_name]+task_info.duration <= self.start_time_vars[task_info.predecessors]  )

        self.model.Minimize(sum(self.start_time_vars[task_name] for task_name, task_info in self.tasks.items()))        
            # Solve the model
        self.solver = cp_model.CpSolver()
        self.status = self.solver.solve(self.model)

    def print_start_date(self):
        # Extract and print the solution
        scheduled_times = {}
        if self.status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            for task_name in self.tasks:
                self.start_time = self.solver.value(self.start_time_vars[task_name])
                scheduled_times[task_name] = self.start_time
                print(f"{task_name} starts at {self.start_time}")
            print(self.solver.objective_value)
        else:
            print("No feasible solution found.")
    
    def print_order(self):
        print("To do")

In [7]:
from ortools.sat.python import cp_model
from collections import namedtuple
# Define task information using namedtuples
taskInfo = namedtuple("taskInfo", ["duration", "predecessors", "relase_date", "due_date"])

# task difinitions
tasks = {
    "task_a_1": taskInfo(
        duration = 10 ,  
        predecessors = "none",
        relase_date = 500,
        due_date = 900
    ),

}
# Machine 
machines = ["m_a"]

Opti_1 = Machine_Parallele( taskInfo, 
    tasks, 
    machines )

Opti_1.print_start_date()

task_a_1 starts at 500
500.0


In [8]:
from ortools.sat.python import cp_model
from collections import namedtuple


# Define task information using namedtuples
taskInfo = namedtuple("taskInfo", ["duration", "predecessors", "relase_date", "due_date"])

# task difinitions
tasks = {
    "task_a_1": taskInfo(
        duration = 20 ,  
        predecessors = "task_a_2",
        relase_date = 500,
        due_date = 600
    ),
    "task_a_2": taskInfo(
        duration = 120 ,  
        predecessors = "none" ,
        relase_date = 500,
        due_date = 900
    ),
    "task_b_1": taskInfo(
        duration = 120 ,  
        predecessors = "task_b_2",
        relase_date = 100,
        due_date = 600
    ),
    "task_b_2": taskInfo(
        duration = 120 ,  
        predecessors = "none",
        relase_date = 0,
        due_date = 600
    ),
}
# Machine 
machines = ["m_a", "m_b"]

Opti_1 = Machine_Parallele( taskInfo, 
    tasks, 
    machines )


Opti_1.print_start_date()

task_a_1 starts at 500
task_a_2 starts at 520
task_b_1 starts at 100
task_b_2 starts at 220
1340.0
